# ETL-17 · Padronização de profissão

- **Etapa:** ETL-17, Padronização de profissão
- **Responsável:** Pedro Wellington
- **Issue:** ETL-17
- **Entrada:** `data/intermediarios/03_texto.parquet`
- **Saída:** `data/intermediarios/06_limpo.parquet`

## 1. Setup

In [57]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

INTERM = RAIZ / "data" / "intermediarios"      # .parquet (o .gitignore já ignora)

ENTRADA = INTERM / "03_texto.parquet"          # saída da ETL-14
SAIDA   = INTERM / "06_limpo.parquet"          # resultado desta etapa

## 2. Carregar a entrada

In [58]:
df = pd.read_parquet(ENTRADA)
print('shape:', df.shape)

# quantas grafias distintas de profissão e as 30 mais comuns
print('grafias distintas:', df['profissao'].nunique())
df['profissao'].value_counts().head(30)

shape: (806977, 33)
grafias distintas: 814


profissao
MOTORISTA                      70309
AUTONOMO(A)                    46750
MOTO-BOY                       43734
APOSENTADO(A)                  36703
ESTUDANTE                      31045
AJUDANTE                       24524
OUTRA                          21544
DO LAR                         17341
VENDEDOR(A)                    17016
DESEMPREGADO(A)                15476
EMPRESARIO(A)                  13890
PROFESSOR(A)                   12956
PEDREIRO                       12955
COMERCIANTE                    12292
MECANICO(A)                    10714
ENTREGADOR(A)                  10465
OPERADOR DE MAQUINA             8860
AUXILIAR ADMINISTRATIVO         8354
AUXILIAR DE SERVIÇOS GERAIS     7701
AUXILIAR GERAL                  7195
VIGILANTE                       7154
POLICIAL MILITAR                7061
ATENDENTE                       6723
ENGENHEIRO(A)                   6681
AUXILIAR DE PRODUCAO            6653
ADMINISTRADOR(A)                6129
GERENTE                     

## 3. Diagnóstico ANTES

In [59]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,float64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


## 4. Transformações

Cada tratamento está numa função. Nenhum deles remove linhas.

### 4.1 Padronizar a escrita

In [60]:
def padronizar_profissao(df: pd.DataFrame) -> pd.DataFrame:
    """Deixa a escrita igual: maiúsculas, sem acento, sem (A), sem hífen e sem espaço duplo."""
    df = df.copy()
    s = df['profissao'].str.upper()

    # tira acentos (Á vira A)
    s = s.str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii')

    s = s.str.replace(r'\(A\)|\(O\)', '', regex=True)   # MECANICO(A) vira MECANICO
    s = s.str.replace(r'[-/]', ' ', regex=True)         # MOTO-BOY vira MOTO BOY
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()  # espaço duplo e pontas

    df['profissao'] = s
    return df

n_antes = len(df); distintas_antes = df['profissao'].nunique()
df = padronizar_profissao(df)
print('linhas:', n_antes, '->', len(df))
print('grafias distintas:', distintas_antes, '->', df['profissao'].nunique())

linhas: 806977 -> 806977
grafias distintas: 814 -> 562


### 4.2 Ver o que ainda sobra (só consulta)

In [61]:
# as 80 profissões mais comuns depois de padronizar (cobrem quase toda a base)
contagem = df['profissao'].value_counts()
print('cobertura das 80 maiores:', round(contagem.head(80).sum() / contagem.sum() * 100, 1), '%')
print(' | '.join(f'{p} ({n})' for p, n in contagem.head(80).items()))

cobertura das 80 maiores: 87.0 %
MOTORISTA (70535) | AUTONOMO (46912) | MOTO BOY (43846) | APOSENTADO (37019) | ESTUDANTE (31220) | AJUDANTE (24595) | OUTRA (21544) | DO LAR (17341) | VENDEDOR (17065) | DESEMPREGADO (15738) | EMPRESARIO (13925) | PEDREIRO (13020) | PROFESSOR (12977) | COMERCIANTE (12333) | MECANICO (10771) | ENTREGADOR (10485) | OPERADOR DE MAQUINA (8874) | AUXILIAR ADMINISTRATIVO (8372) | AUXILIAR DE SERVICOS GERAIS (7746) | AUXILIAR GERAL (7202) | VIGILANTE (7184) | POLICIAL MILITAR (7090) | ATENDENTE (6746) | ENGENHEIRO (6690) | AUXILIAR DE PRODUCAO (6677) | ADMINISTRADOR (6140) | GERENTE (6139) | ELETRICISTA (6036) | TECNICO (5978) | ENFERMEIRO (5852) | PINTOR (5021) | ANALISTA (4818) | OPERADOR (4731) | ADVOGADO (4603) | FUNCION.PUBLICO MUNICIPAL (4014) | ASSISTENTE ADMINISTRATIVO (3954) | DOMESTICA (3877) | PORTEIRO (3843) | AUXILIAR DE ENFERMAGEM (3810) | METALURGICO (3451) | COZINHEIRO (3383) | ACOUGUEIRO (3373) | BALCONISTA (3182) | SEGURANCA (3106) | CABELEIR

### 4.3 Unificar grafias que ainda diferem

In [62]:
# grafias que significam a mesma coisa e a padronização de escrita não juntou
UNIFICAR = {
    'MOTOBOY': 'MOTO BOY',
    'MOTOFRETISTA': 'MOTO FRETISTA',
}

df['profissao'] = df['profissao'].replace(UNIFICAR)
print('grafias distintas:', df['profissao'].nunique())

grafias distintas: 561


### 4.4 Criar o grupo da profissão

In [63]:
# cada grupo tem palavras-chave; vale o primeiro grupo que combinar (a ordem importa)
REGRAS = [
    ('SEM OCUPACAO', r'ESTUDANTE|APOSENTAD|DESEMPREGAD|DESOCUPAD|DO LAR|PENSIONISTA'),
    ('SAUDE E EDUCACAO', r'ENFERM|MEDIC|FISIOTERAP|DENTISTA|PSICOLOG|FARMAC|NUTRICION|VETERINAR|PROFESSOR|PEDAGOG|INSTRUTOR'),
    ('TRANSPORTE E ENTREGA', r'MOTORISTA|MOTO ?BOY|MOTO ?FRETISTA|MOTOTAXISTA|MOTOCICLISTA|ENTREGADOR|TAXISTA|CAMINHONEIRO|CONDUTOR|FRETISTA|COBRADOR|MANOBRISTA'),
    ('SEGURANCA E SERVICO PUBLICO', r'VIGILANTE|POLICIAL|SEGURANCA|PORTEIRO|BOMBEIRO|MILITAR|FUNCION|SERVIDOR PUBLICO|GUARDA'),
    ('TECNICOS E NIVEL SUPERIOR', r'ENGENHEIR|ADVOGAD|ARQUITET|ANALISTA DE SISTEMAS|PROGRAMADOR|TECNICO|DESIGNER|JORNALISTA|PUBLICITARIO'),
    ('ADMINISTRATIVO', r'ADMINISTRAT|ADMINISTRADOR|ANALISTA|SUPERVISOR|GERENTE|CONTADOR|SECRETARI|ENCARREGADO|BANCARI|ASSISTENTE|COORDENADOR|DIRETOR|RECEPCIONISTA|ESTAGIARIO|AUXILIAR DE ESCRITORIO|ALMOXARIFE|ESTOQUISTA|CONFERENTE|CONSULTOR'),
    ('COMERCIO E VENDAS', r'VENDEDOR|COMERCIANTE|REPOSITOR|REPRESENTANTE|PROMOTOR|CAIXA|BALCONISTA|ATENDENTE|COMERCIAL|CORRETOR'),
    ('AGRICULTURA E CAMPO', r'AGRICULTOR|LAVRADOR|TRATORISTA|PECUARIST|JARDINEIR|CAMPO'),
    ('INDUSTRIA E CONSTRUCAO', r'PEDREIRO|PINTOR|MECANICO|OPERADOR|METALURG|SOLDADOR|SERRALHEIR|MONTADOR|ELETRICISTA|PRODUCAO|MARCENEIR|SERVENTE|AJUDANTE|CARPINTEIR|ENCANADOR|MANUTENCAO|OPERARIO|COSTUREIR|FUNILEIR|SAPATEIR|VIDRACEIR'),
    ('SERVICOS PESSOAIS E DOMESTICOS', r'AUXILIAR GERAL|SERVICOS GERAIS|DOMESTIC|DIARISTA|LIMPEZA|COZINH|CABELEIREIR|MANICURE|BARBEIR|GARCOM|GARCON|ACOUGUEIR|FRENTISTA|FAXINEIR|CUIDADOR'),
    ('AUTONOMO E EMPRESARIO', r'AUTONOMO|EMPRESARIO|PROPRIETARIO'),
]

def classificar_grupos(profissao: pd.Series) -> pd.Series:
    """Começa tudo como OUTROS e preenche os grupos de trás para frente, para valer o primeiro."""
    grupo = pd.Series('OUTROS', index=profissao.index)
    for nome, padrao in reversed(REGRAS):
        grupo[profissao.str.contains(padrao, na=False)] = nome
    return grupo

df['grupo_profissao'] = classificar_grupos(df['profissao'])
df.loc[df['profissao'].isna(), 'grupo_profissao'] = 'NÃO INFORMADO'

print(df['grupo_profissao'].value_counts())
# o que ficou em OUTROS: serve para achar palavras-chave que faltam
print(df.loc[df['grupo_profissao'] == 'OUTROS', 'profissao'].value_counts().head(25))

grupo_profissao
TRANSPORTE E ENTREGA              131023
INDUSTRIA E CONSTRUCAO            106781
SEM OCUPACAO                      102987
NÃO INFORMADO                      85900
OUTROS                             62275
AUTONOMO E EMPRESARIO              60892
ADMINISTRATIVO                     58429
COMERCIO E VENDAS                  50758
SERVICOS PESSOAIS E DOMESTICOS     44795
SAUDE E EDUCACAO                   35643
SEGURANCA E SERVICO PUBLICO        32054
TECNICOS E NIVEL SUPERIOR          28996
AGRICULTURA E CAMPO                 6444
Name: count, dtype: int64
profissao
OUTRA                   21544
INSTALADOR               1146
PADEIRO                  1040
FISCAL                   1010
ESTETICISTA               910
CONSTRUTOR                892
AGENTE PENITENCIARIO      891
INSPETOR                  890
PIZZAIOLO                 868
LAVADOR DE CARRO          843
ZELADOR                   821
BABA                      781
BORRACHEIRO               772
AGENTE DE APOIO          

## 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [64]:
depois = diagnostico(df)
depois

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,float64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


## 6. Salvar a saída

In [65]:
df.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA, '| shape:', df.shape)

salvo em: c:\Users\pedro\Downloads\Projeto_integração_bd\data\intermediarios\06_limpo.parquet | shape: (806977, 34)


## 7. Resumo

**O que foi feito:** padronização da escrita das profissões e criação da coluna `grupo_profissao`. O resultado foi salvo em `data/intermediarios/06_limpo.parquet`.

**Padronização:** maiúsculas, sem acento, sem "(A)" e sem hífen (MOTO-BOY virou MOTO BOY). Duas grafias foram unificadas à mão (MOTOBOY e MOTOFRETISTA). As grafias distintas caíram de 814 para 561.

**Grupos:** a profissão recebe um de 11 grupos por palavras-chave, como TRANSPORTE E ENTREGA, INDUSTRIA E CONSTRUCAO e SAUDE E EDUCACAO. Profissões sem palavra-chave ficam em OUTROS (62.275 linhas, 7,7%), a maior parte da própria categoria OUTRA da fonte (21.544). Quem não tem profissão informada fica em NÃO INFORMADO (85.900 linhas).

**O que não mudou:** nenhuma linha foi removida (806.977). A coluna `profissao` continua com 10,64% de nulos, que a `dim_profissao` trata com o membro -1. A tabela final tem 34 colunas.

**Observação:** os grupos usam palavras-chave e podem ser refinados. Os grupos finais seguem a ordem das regras, e o primeiro que combina vale.